# 📊 Day 1, Topic 6 — Basic Column Operations: Full Toolkit

This topic covers the essential column-level operations you'll use in nearly every data project:

| Operation | Method / Syntax |
|-----------|----------------|
| Change dtype | `.astype(type)` |
| Create new column | `df['col'] = expression` |
| Boolean column | `df['col'] = df['other'] > threshold` |
| Map values | `.map(dict)` |
| Conditional values | `np.where(condition, if_true, if_false)` |
| Rename columns | `.rename(columns={old: new})` |
| Drop columns | `.drop(['col'], axis=1)` |
| Sort rows | `.sort_values('col', ascending=True/False)` |
| Filter rows | `df[boolean_mask]` |
| Filter with SQL-like syntax | `.query("condition string")` |


Day 1, Topic 6: Basic Column Operations – Full Toolkit

In [10]:
import pandas as pd
import numpy as np

## `.astype()` — Converting Data Types

```python
df['col'].astype(target_type)
```

Common conversions:
| From | To | Why |
|------|----|-----|
| `int64` (0/1) | `bool` | Make logic explicit |
| `int64` | `str` / `object` | For display or joining as a key |
| `object` (numbers as strings) | `float64` | Enable math operations |
| `float64` | `int64` | Remove decimal if no fractional data |

> ⚠️ Columns with `NaN` can't be cast to plain `int64` — use `Int64` (nullable integer) or fill NaNs first.


In [3]:
# Converting Data Types with .astype()
df = pd.DataFrame({
    'ID': [1, 2, 3],
    'Score': [85.5, 92.0, 78.3],
    'Passed': [1, 1, 0]
    })

df['Passed'] = df['Passed'].astype(bool)
df['ID'] = df['ID'].astype(str)

print(df.dtypes)

ID         object
Score     float64
Passed       bool
dtype: object


## Creating New Columns — Vectorized Assignment

```python
df['NewCol'] = df['A'] * df['B']   # derived from other columns
df['Constant'] = 'USD'             # scalar broadcasts to all rows
```

This is always vectorized — it runs in C, not Python loops.  
The new column is added at the rightmost position of the DataFrame.


In [4]:
#Creating New Columns (Vectorized)
df = pd.DataFrame({
    'Quantity': [5, 3, 10],
    'Price': [100, 50, 20]
})

df['Total'] = df['Quantity'] * df['Price']
df['Currency'] = 'USD'
print(df)


   Quantity  Price  Total Currency
0         5    100    500      USD
1         3     50    150      USD
2        10     20    200      USD


## Boolean Columns

```python
df['IsExpensive'] = df['Price'] > 40   # True/False per row
```

Boolean columns are extremely useful for:
- Quick filtering: `df[df['IsExpensive']]`
- Counting: `df['IsExpensive'].sum()` (True = 1, False = 0)
- Aggregating by flag: `df.groupby('IsExpensive')['Revenue'].mean()`


In [6]:
#Creating Boolean columns:
df['IsExpensive'] = df['Price'] > 40
print(df)

   Quantity  Price  Total Currency  IsExpensive
0         5    100    500      USD         True
1         3     50    150      USD         True
2        10     20    200      USD        False


## Modifying Existing Columns + `.map()` + `np.where()`

### In-place modification
```python
df['Price'] = df['Price'] * 1.1   # 10% price increase
```

### `.map(dict)` — Replace values using a lookup table
```python
df['Category'] = df['Category'].map({'A': 'Alpha', 'B': 'Bravo'})
```
Any value not in the dict becomes `NaN`. Useful for encoding categories or translating codes to labels.

### `np.where(condition, value_if_true, value_if_false)` — Conditional column
```python
df['Status'] = np.where(df['Price'] > 40, 'Expensive', 'Cheap')
```
This is the vectorized equivalent of `if/else` — much faster than `.apply(lambda ...)`.


In [12]:
#Modifying Existing Columns
df['Price'] = df['Price'] * 1.1

df['Category'] = pd.Series(['A', 'B', 'C'])
df['Category'] = df['Category'].map({'A': 'Alpha', 'B': 'Bravo', 'C': 'Charlie'})

df['Status'] = np.where(df['Price'] > 40, 'Expensive', 'Cheap')
print(df)

   Quantity     Price  Total Currency  IsExpensive Category     Status
0         5  161.0510    500      USD         True    Alpha  Expensive
1         3   80.5255    150      USD         True    Bravo  Expensive
2        10   32.2102    200      USD        False  Charlie      Cheap


## `.rename()` — Fixing Column Names

```python
df = df.rename(columns={'old_name': 'new_name', 'another': 'better'})
```

Always assign the result back (or use `inplace=True`).  
Use this to:
- Fix typos in column names
- Make abbreviations readable (`'nm'` → `'Name'`)
- Standardize naming conventions (all lowercase, no spaces)


In [15]:
#Renaming Columns with .rename()

df = pd.DataFrame({
    'nm': ['Alice', 'Bob'],
    'ag': [25, 30]
})

df = df.rename(columns={'nm':'Name', 'ag':'Age'})
print(df)

    Name  Age
0  Alice   25
1    Bob   30


## `.drop()` — Removing Columns (or Rows)

```python
df = df.drop('B', axis=1)           # drop one column
df = df.drop(['B', 'C'], axis=1)    # drop multiple columns
df = df.drop([0, 2], axis=0)        # drop rows by index label
```

`axis=1` means "columns" (the horizontal axis).  
`axis=0` means "rows" (the vertical axis).

> **Memory tip:** Dropping unneeded columns before processing reduces RAM usage and speeds up subsequent operations.


In [18]:
#Dropping Columns with .drop()

df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6],
    'C': [7, 8, 9]
})

df = df.drop('B', axis=1)
print(df)

   A  C
0  1  7
1  2  8
2  3  9


## `.sort_values()` — Reordering Rows

```python
df.sort_values('col', ascending=True)         # smallest → largest
df.sort_values('col', ascending=False)        # largest → smallest
df.sort_values(['col1', 'col2'])              # sort by multiple columns
```

The original index is preserved (just reordered). Use `.reset_index(drop=True)` afterward if you want a clean 0-N index.


In [22]:
#Sorting Values with .sort_values()
df = pd.DataFrame({
    'Name': ['Faizan', 'Ali', 'Rohan'],
    'Scores': [88, 92, 85]
})

df = df.sort_values('Scores')
print(df)

     Name  Scores
2   Rohan      85
0  Faizan      88
1     Ali      92


## Boolean Filtering (Masking) — Select Rows

```python
df[df['Age'] > 20]                              # single condition
df[(df['Age'] > 20) & (df['Salary'] > 25000)]  # AND
df[(df['Age'] < 20) | (df['Salary'] > 90000)]  # OR
```

**Rules:**
- Each condition must be in its own `()`
- Use `&` not `and`, `|` not `or`
- Use `~` for NOT: `df[~df['IsAlone']]`


In [29]:
#Boolean Filtering (Masking)
df = pd.DataFrame({
    'Name': ['Faizan', 'Ali', 'Rohan', 'Arslan', 'Gull'],
    'Age': [18, 15, 22, 23, 25],
    'Salary': [50000, 100000, 45000, 32000, 45100]
})

older = df[df['Age'] > 20]
print(older)

filtered = df[(df['Age'] > 20) & (df['Salary'] > 25000)]
print(filtered)

     Name  Age  Salary
2   Rohan   22   45000
3  Arslan   23   32000
4    Gull   25   45100
     Name  Age  Salary
2   Rohan   22   45000
3  Arslan   23   32000
4    Gull   25   45100


## `.query()` — SQL-Style Filtering

```python
df.query("Age > 20 and Salary > 25000")
```

`.query()` is a cleaner syntax for complex filters — especially when column names don't have spaces.  
It produces the same result as the bracket-mask approach.  
You can reference Python variables with `@var`: `df.query("Age > @min_age")`.


In [30]:
filtered_2 = df.query("Age > 20 and Salary > 25000")
print(filtered)

     Name  Age  Salary
2   Rohan   22   45000
3  Arslan   23   32000
4    Gull   25   45100


---
## 📝 Activity — Titanic Column Engineering

This activity applies all six techniques to a real dataset in one workflow:
1. `astype(bool)` — convert Survived
2. Arithmetic → new column `FamilySize`
3. Boolean comparison → `IsAlone`
4. `np.where()` → `SexCode` (male=0, female=1)
5. `.rename()` → standardize column names
6. `.drop()` → remove irrelevant columns
7. `.sort_values()` → find highest-fare passengers
8. Boolean filter → count female survivors


In [36]:
#Activity
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# 1. Convert Survived to bool
df['Survived'] = df['Survived'].astype(bool)

# 2. Create FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# 3. Create IsAlone
df['IsAlone'] = df['FamilySize'] == 1

# 4. Create SexCode (male=0, female=1)
df['SexCode'] = np.where(df['Sex'] == 'male', 0, 1)

# 5. Rename columns
df = df.rename(columns={
    'Pclass': 'PassengerClass',
    'SibSp': 'SiblingsSpouses',
    'Parch': 'ParentsChildren'
})

# 6. Drop unnecessary columns
df = df.drop(['Ticket', 'Cabin'], axis=1)

# 7. Sort by Fare (descending)
top_fares = df.sort_values('Fare', ascending=False).head(5)
print(top_fares[['Name', 'Fare']])

# 8. Filter: female survivors
female_survivors = df[(df['Sex'] == 'female') & (df['Survived'] == True)]
print(f"Female survivors: {len(female_survivors)}")

                                   Name      Fare
679  Cardeza, Mr. Thomas Drake Martinez  512.3292
258                    Ward, Miss. Anna  512.3292
737              Lesurer, Mr. Gustave J  512.3292
88           Fortune, Miss. Mabel Helen  263.0000
438                   Fortune, Mr. Mark  263.0000
Female survivors: 233
